# 🧮 Vector Embeddings và Đo lường Độ Tương Đồng Ngữ Nghĩa

## Mục tiêu bài học
- Hiểu khái niệm Vector Embeddings và cách biểu diễn ngữ nghĩa trong không gian vector
- Nắm vững các phương pháp đo lường độ tương đồng:
  - Cosine Similarity
  - Euclidean Distance
  - Dot Product
- Áp dụng vào bài toán thực tế với văn bản tiếng Việt và tiếng Anh
- So sánh ưu nhược điểm của từng phương pháp

In [1]:
# Cài đặt các thư viện cần thiết
%pip install sentence-transformers numpy matplotlib scikit-learn pandas gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

### Load Pre-trained Embedding Model

Chúng ta sẽ sử dụng **sentence-transformers**, một thư viện mạnh mẽ cho embeddings:
- Model: `paraphrase-multilingual-MiniLM-L12-v2` - hỗ trợ 50+ ngôn ngữ, bao gồm tiếng Việt
- Kích thước vector: 384 chiều
- Nhanh và hiệu quả cho hầu hết các ứng dụng

In [3]:
# Load model đa ngôn ngữ
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"✅ Model loaded successfully!")
print(f"📊 Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
📊 Embedding dimension: 384


## 🔬 Phần 3: Tạo Vector Embeddings

### 3.1 Embeddings cho từ và câu đơn giản

In [4]:
# Tạo embeddings cho các câu tiếng Anh
sentences_en = [
    "I love programming",
    "I enjoy coding",
    "The weather is nice today",
    "Machine learning is fascinating",
    "Deep learning is a subset of AI"
]

# Tạo embeddings
embeddings_en = model.encode(sentences_en)

print("=" * 70)
print("EMBEDDINGS TIẾNG ANH")
print("=" * 70)
for i, sentence in enumerate(sentences_en):
    print(f"\nCâu {i+1}: {sentence}")
    print(f"First 10 values: {embeddings_en[i][:10]}")
    print(f"Vector norm: {np.linalg.norm(embeddings_en[i]):.4f}")

EMBEDDINGS TIẾNG ANH

Câu 1: I love programming
First 10 values: [-0.09754957 -0.37010992 -0.04942968 -0.11293895 -0.24742271 -0.15186767
  0.0716017   0.17737548  0.0095045   0.5964115 ]
Vector norm: 5.6580

Câu 2: I enjoy coding
First 10 values: [ 0.05587781 -0.4196714  -0.2494934   0.00381995 -0.311845    0.30276927
  0.30298117  0.27343205 -0.21720481  0.7214778 ]
Vector norm: 5.5543

Câu 3: The weather is nice today
First 10 values: [ 0.28088745  0.27709776 -0.07001772  0.08150014  0.24186812  0.09056901
  0.5559215  -0.10870065 -0.58855444  0.13057   ]
Vector norm: 4.9209

Câu 4: Machine learning is fascinating
First 10 values: [-0.1311065  -0.15423486 -0.27258596 -0.32290596 -0.09494047 -0.10874902
 -0.06425487 -0.00107528 -0.03084948 -0.08673868]
Vector norm: 4.9938

Câu 5: Deep learning is a subset of AI
First 10 values: [-0.1499104  -0.17846772 -0.0457218   0.00266017 -0.04144255  0.3145865
 -0.00095411 -0.27811605  0.27148744 -0.08658599]
Vector norm: 4.6359


In [5]:
# Tạo embeddings cho các câu tiếng Việt
sentences_vi = [
    "Tôi yêu lập trình",
    "Tôi thích viết code",
    "Hôm nay thời tiết đẹp",
    "Học máy rất hấp dẫn",
    "Học sâu là một phần của AI"
]

embeddings_vi = model.encode(sentences_vi)

print("=" * 70)
print("EMBEDDINGS TIẾNG VIỆT")
print("=" * 70)
for i, sentence in enumerate(sentences_vi):
    print(f"\nCâu {i+1}: {sentence}")
    print(f"First 10 values: {embeddings_vi[i][:10]}")
    print(f"Vector norm: {np.linalg.norm(embeddings_vi[i]):.4f}")

EMBEDDINGS TIẾNG VIỆT

Câu 1: Tôi yêu lập trình
First 10 values: [-0.19852489 -0.2755765  -0.15470271 -0.1960664  -0.21292336 -0.15973477
  0.125484    0.18644778  0.08417963  0.400628  ]
Vector norm: 4.6289

Câu 2: Tôi thích viết code
First 10 values: [-0.24222176 -0.29427284 -0.35194948  0.02093121 -0.2969828   0.18962647
 -0.04083517  0.20632535 -0.1363573   0.56801367]
Vector norm: 4.7459

Câu 3: Hôm nay thời tiết đẹp
First 10 values: [ 0.2985106   0.3121923   0.07353804  0.15156218  0.3158772   0.02751107
  0.5727888  -0.1669505  -0.5274005   0.18856232]
Vector norm: 4.7281

Câu 4: Học máy rất hấp dẫn
First 10 values: [ 0.06369287  0.04186537 -0.30384415  0.02418213 -0.3230305  -0.19876227
  0.00148263  0.03878158  0.04176662  0.13828395]
Vector norm: 3.8851

Câu 5: Học sâu là một phần của AI
First 10 values: [ 0.1153677  -0.11437081 -0.1235828  -0.01636968 -0.09540617  0.16901153
  0.20097955 -0.06655237  0.34352303  0.11504236]
Vector norm: 4.2875


### So sánh giữa Static vs Contextual embedding 

In [6]:
import numpy as np
import gensim.downloader as api

print("🔄 Loading GloVe model (static embedding)...")
glove = api.load('glove-wiki-gigaword-50')  # 50-dim, ~66MB
print("✅ GloVe loaded!\n")

# Static embedding: "bank" trong ngữ cảnh tài chính và tự nhiên
# → GloVe chỉ là lookup table, không quan tâm câu xung quanh
sentence_finance = "I went to the bank to deposit my money"
sentence_nature  = "The river bank was covered with green grass"

# Dù 2 câu hoàn toàn khác nghĩa, GloVe vẫn trả về CÙNG 1 VECTOR cho "bank"
static_vec_finance = glove["bank"]  # vector của "bank" trong câu tài chính
static_vec_nature  = glove["bank"]  # vector của "bank" trong câu tự nhiên → giống hệt

print("📌 STATIC EMBEDDING (GloVe)")
print(f"\nCâu 1: '{sentence_finance}'")
print(f"Vector 'bank' (10 dim đầu): {static_vec_finance[:10].round(4)}")

print(f"\nCâu 2: '{sentence_nature}'")
print(f"Vector 'bank' (10 dim đầu): {static_vec_nature[:10].round(4)}")

print(f"\n⚠️  Hai vector hoàn toàn GIỐNG HỆT nhau vì GloVe không đọc ngữ cảnh câu")

🔄 Loading GloVe model (static embedding)...
✅ GloVe loaded!

📌 STATIC EMBEDDING (GloVe)

Câu 1: 'I went to the bank to deposit my money'
Vector 'bank' (10 dim đầu): [ 0.6649 -0.1139  0.6784  0.1795  0.6828 -0.4779 -0.3076  0.1749 -0.7051
 -0.5502]

Câu 2: 'The river bank was covered with green grass'
Vector 'bank' (10 dim đầu): [ 0.6649 -0.1139  0.6784  0.1795  0.6828 -0.4779 -0.3076  0.1749 -0.7051
 -0.5502]

⚠️  Hai vector hoàn toàn GIỐNG HỆT nhau vì GloVe không đọc ngữ cảnh câu


In [7]:
from transformers import AutoTokenizer, AutoModel
import torch

print("🔄 Loading BERT model (contextual embedding)...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased")
bert_model.eval()
print("✅ BERT loaded!\n")

def get_word_vector_in_context(sentence, target_word, tokenizer, model):
    """Lấy vector của target_word trong ngữ cảnh câu (contextual)."""
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0]  # (seq_len, 768)

    # Tìm index các sub-tokens của target_word
    indices = [i for i, t in enumerate(tokens) if target_word.lower() in t.lower()]
    if not indices:
        raise ValueError(f"'{target_word}' không tìm thấy trong tokens: {tokens}")

    # Average nếu target_word bị tách thành nhiều sub-tokens
    return hidden[indices].mean(dim=0).numpy()

# Cùng từ "bank" — 2 nghĩa hoàn toàn khác nhau
sentence_finance = "I went to the bank to deposit my money"
sentence_nature  = "The river bank was covered with green grass"

vec_finance = get_word_vector_in_context(sentence_finance, "bank", tokenizer, bert_model)
vec_nature  = get_word_vector_in_context(sentence_nature,  "bank", tokenizer, bert_model)

print("📌 CONTEXTUAL EMBEDDING (BERT)")
print(f"\nCâu 1: '{sentence_finance}'")
print(f"Vector 'bank' (10 dim đầu): {vec_finance[:10].round(4)}")

print(f"\nCâu 2: '{sentence_nature}'")
print(f"Vector 'bank' (10 dim đầu): {vec_nature[:10].round(4)}")

🔄 Loading BERT model (contextual embedding)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERT loaded!

📌 CONTEXTUAL EMBEDDING (BERT)

Câu 1: 'I went to the bank to deposit my money'
Vector 'bank' (10 dim đầu): [ 0.2313 -0.0412 -0.0569  0.0196  0.7849 -0.0544 -0.2636  0.764  -0.2974
  0.2072]

Câu 2: 'The river bank was covered with green grass'
Vector 'bank' (10 dim đầu): [-0.0634 -0.2577 -0.1702 -0.0874  0.258   0.4441 -0.1707  0.995   0.105
 -0.2856]


In [8]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Static: 2 câu khác nghĩa nhưng cùng từ "bank" → cùng vector → similarity = 1.0
static_sim     = cosine_sim(static_vec_finance, static_vec_nature)
# Contextual: BERT tạo vector khác nhau theo ngữ cảnh → similarity < 1.0
contextual_sim = cosine_sim(vec_finance, vec_nature)

print("=" * 60)
print("📊 KẾT QUẢ SO SÁNH")
print("=" * 60)
print(f"\nCâu 1: 'I went to the BANK to deposit my money'  (ngân hàng)")
print(f"Câu 2: 'The river BANK was covered with green grass'  (bờ sông)")

print(f"\n🔵 STATIC (GloVe)")
print(f"   Cosine Similarity: {static_sim:.4f}  ← = 1.0, không phân biệt được 2 nghĩa")

print(f"\n🟢 CONTEXTUAL (BERT)")
print(f"   Cosine Similarity: {contextual_sim:.4f}  ← < 1.0, BERT phân biệt được 2 nghĩa")

print(f"\n💡 NHẬN XÉT:")
print(f"   Static     → lookup table cố định: 'bank' luôn = 1 vector duy nhất")
print(f"   Contextual → Transformer đọc toàn bộ câu: vector thay đổi theo ngữ cảnh")
print(f"   Similarity BERT = {contextual_sim:.4f} → 2 nghĩa của 'bank' đã được phân tách")

📊 KẾT QUẢ SO SÁNH

Câu 1: 'I went to the BANK to deposit my money'  (ngân hàng)
Câu 2: 'The river BANK was covered with green grass'  (bờ sông)

🔵 STATIC (GloVe)
   Cosine Similarity: 1.0000  ← = 1.0, không phân biệt được 2 nghĩa

🟢 CONTEXTUAL (BERT)
   Cosine Similarity: 0.6773  ← < 1.0, BERT phân biệt được 2 nghĩa

💡 NHẬN XÉT:
   Static     → lookup table cố định: 'bank' luôn = 1 vector duy nhất
   Contextual → Transformer đọc toàn bộ câu: vector thay đổi theo ngữ cảnh
   Similarity BERT = 0.6773 → 2 nghĩa của 'bank' đã được phân tách


## 📐 Phần 4: Các Phép Đo Độ Tương Đồng

### 4.1 Cosine Similarity (Độ tương đồng Cosine)

**Công thức:**
$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|} = \frac{\sum_{i=1}^{n} A_i \times B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \times \sqrt{\sum_{i=1}^{n} B_i^2}}$$

**Giá trị:**
- Range: [-1, 1]
- 1: Hoàn toàn giống nhau (cùng hướng)
- 0: Không liên quan (vuông góc)
- -1: Hoàn toàn ngược nhau

**Đặc điểm:**
- ✅ Không phụ thuộc vào độ lớn của vector
- ✅ Chỉ quan tâm đến hướng/góc giữa các vector
- ✅ Phổ biến nhất trong NLP và Recommendation Systems
- ❌ Không tính đến magnitude

In [9]:
# Test với 2 câu đầu tiên (tiếng Anh)
vec1 = embeddings_en[0]  # "I love programming"
vec2 = embeddings_en[1]  # "I enjoy coding"
vec3 = embeddings_en[2]  # "The weather is nice today"

In [10]:
# Implement Cosine Similarity từ đầu
def cosine_similarity_manual(vec1, vec2):
    """
    Tính cosine similarity giữa 2 vectors
    """
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    
    return dot_product / (norm_vec1 * norm_vec2)



print("=" * 70)
print("COSINE SIMILARITY - TIẾNG ANH")
print("=" * 70)

sim_1_2 = cosine_similarity_manual(vec1, vec2)
sim_1_3 = cosine_similarity_manual(vec1, vec3)

print(f"\n'{sentences_en[0]}' vs '{sentences_en[1]}'")
print(f"Cosine Similarity: {sim_1_2:.4f}")

print(f"\n'{sentences_en[0]}' vs '{sentences_en[2]}'")
print(f"Cosine Similarity: {sim_1_3:.4f}")

print(f"\n💡 Nhận xét: Câu 1 và 2 có ý nghĩa tương tự → similarity cao ({sim_1_2:.4f})")
print(f"💡 Nhận xét: Câu 1 và 3 khác nghĩa → similarity thấp ({sim_1_3:.4f})")

COSINE SIMILARITY - TIẾNG ANH

'I love programming' vs 'I enjoy coding'
Cosine Similarity: 0.8598

'I love programming' vs 'The weather is nice today'
Cosine Similarity: 0.1603

💡 Nhận xét: Câu 1 và 2 có ý nghĩa tương tự → similarity cao (0.8598)
💡 Nhận xét: Câu 1 và 3 khác nghĩa → similarity thấp (0.1603)


### 4.2 Euclidean Distance (Khoảng cách Euclidean)

**Công thức:**
$$\text{euclidean\_distance}(A, B) = \sqrt{\sum_{i=1}^{n} (A_i - B_i)^2}$$

**Giá trị:**
- Range: [0, ∞]
- 0: Hai vector giống hệt nhau
- Càng lớn: Càng khác nhau

**Đặc điểm:**
- ✅ Đơn giản, trực quan
- ✅ Tính đến cả magnitude và direction
- ❌ Phụ thuộc vào độ lớn của vector
- ❌ Nhạy cảm với scale của dữ liệu
- ❌ Không hiệu quả với high-dimensional data (curse of dimensionality)

In [11]:
# Implement Euclidean Distance từ đầu
def euclidean_distance_manual(vec1, vec2):
    """
    Tính Euclidean distance giữa 2 vectors
    """
    return np.sqrt(np.sum((vec1 - vec2) ** 2))

print("=" * 70)
print("EUCLIDEAN DISTANCE - TIẾNG ANH")
print("=" * 70)

dist_1_2 = euclidean_distance_manual(vec1, vec2)
dist_1_3 = euclidean_distance_manual(vec1, vec3)

print(f"\n'{sentences_en[0]}' vs '{sentences_en[1]}'")
print(f"Euclidean Distance: {dist_1_2:.4f}")

print(f"\n'{sentences_en[0]}' vs '{sentences_en[2]}'")
print(f"Euclidean Distance: {dist_1_3:.4f}")

print(f"\n💡 Nhận xét: Câu 1 và 2 tương tự → distance nhỏ ({dist_1_2:.4f})")
print(f"💡 Nhận xét: Câu 1 và 3 khác nhau → distance lớn ({dist_1_3:.4f})")

EUCLIDEAN DISTANCE - TIẾNG ANH

'I love programming' vs 'I enjoy coding'
Euclidean Distance: 2.9699

'I love programming' vs 'The weather is nice today'
Euclidean Distance: 6.8776

💡 Nhận xét: Câu 1 và 2 tương tự → distance nhỏ (2.9699)
💡 Nhận xét: Câu 1 và 3 khác nhau → distance lớn (6.8776)


### 4.3 Dot Product (Tích vô hướng)

**Công thức:**
$$\text{dot\_product}(A, B) = \sum_{i=1}^{n} A_i \times B_i = A_1 \times B_1 + A_2 \times B_2 + ... + A_n \times B_n$$

**Giá trị:**
- Range: (-∞, ∞)
- Càng lớn: Càng tương đồng
- Giá trị âm: Vectors ngược hướng

**Đặc điểm:**
- ✅ Tính toán đơn giản, nhanh nhất
- ✅ Tính đến cả magnitude và direction
- ✅ Hiệu quả cho normalized vectors
- ❌ Phụ thuộc vào độ lớn của vector
- ❌ Khó so sánh giữa các cặp vectors khác nhau

**Lưu ý:** Nếu vectors được normalize (unit vectors), dot product = cosine similarity!

In [12]:
# Implement Dot Product từ đầu
def dot_product_manual(vec1, vec2):
    """
    Tính dot product giữa 2 vectors
    """
    return np.dot(vec1, vec2)

print("=" * 70)
print("DOT PRODUCT - TIẾNG ANH")
print("=" * 70)

dp_1_2 = dot_product_manual(vec1, vec2)
dp_1_3 = dot_product_manual(vec1, vec3)

print(f"\n'{sentences_en[0]}' vs '{sentences_en[1]}'")
print(f"Dot Product: {dp_1_2:.4f}")

print(f"\n'{sentences_en[0]}' vs '{sentences_en[2]}'")
print(f"Dot Product: {dp_1_3:.4f}")

print(f"\n💡 Nhận xét: Câu 1 và 2 tương tự → dot product cao ({dp_1_2:.4f})")
print(f"💡 Nhận xét: Câu 1 và 3 khác nhau → dot product thấp hơn ({dp_1_3:.4f})")

DOT PRODUCT - TIẾNG ANH

'I love programming' vs 'I enjoy coding'
Dot Product: 27.0219

'I love programming' vs 'The weather is nice today'
Dot Product: 4.4634

💡 Nhận xét: Câu 1 và 2 tương tự → dot product cao (27.0219)
💡 Nhận xét: Câu 1 và 3 khác nhau → dot product thấp hơn (4.4634)
